# 笔记书写约定（Q&A 格式）

与 `笔记/01笔记.ipynb` 对齐，便于回顾与检索：

- **每个题目大类**：单独占一个 markdown cell，且以 **一级标题 `# …`** 开头（VS Code / Jupyter 大纲里一眼能跳）。
- **大类下的具体问题**：用 **`## 1.`、`## 2.`** 或 **`## 具体问题短标题`**，需要再细分时用 `###`。
- **新增内容**：一律在**文末追加** cell，不删改前面已写的问题与结论，方便对照「当时怎么想的」。

后续 Q&A 按此约定追加即可。

> **用途**：个人知识点 Q&A（非交付文档）。  
# 02 数据处理 — 数据质量问题说明

> 记录时间：2026-05-15  
> 数据来源：01 旧 jsonl 上的诊断；02 已用 `parse_pmc.py` 重生成（见 schedule 阶段 3 进度）。  
> 发现方式：`med-data-EDA.ipynb` §0 / §2 跑通后的全量诊断

本文说明当前样本中已确认的**数据质量问题**：成因、影响、修复方式与推荐方案。


## ✅ 更新（2026-05-15）：已在 02 工程内修复

**结论**：问题来自 01 的 `parse_pmc_xml()` XPath，而非 PMC 源数据本身。02 已实现独立数据流，**全量期无需回 01**。

| 模块 | 路径 | 作用 |
|---|---|---|
| 解析器 | `02 数据处理/src/parse_pmc.py` | 修正 title / pub_year / pub_date / pmid / abstract |
| 批量构建 | `02 数据处理/src/build_jsonl.py` | XML 目录 → jsonl |
| 入口脚本 | `02 数据处理/scripts/build_jsonl.sh` | 命令行一键重跑 |

```bash
cd "02 数据处理"
./scripts/build_jsonl.sh --pmcids-from data/processed/sample.jsonl.bak01
```

**重跑后（100 篇）**：title>500 字符 **0 篇**；pub_year 正常；已含 `pmid`、`pub_date`；abstract 仍缺 3 篇（§3 定策略）。

下文「问题 1~4」保留为**历史记录**（01 旧 jsonl 上的诊断）。

## 问题总览

| # | 问题 | 规模（100 篇） | 严重程度 | 是否阻塞 §3 |
|---|---|---|---|---|
| 1 | `title` 被参考文献标题污染 | 96 篇 title > 500 字符 | **高** | 否（§3 应记录） |
| 2 | `pub_year` 多年份拼接 | 85 篇格式异常 | **中** | 否 |
| 3 | `abstract` 缺失 | 3 篇（3%） | **中** | 否（§3 定策略） |
| 4 | `pmid` 未抽取 | 100 篇均无 | **低**（任务书可选） | 否 |

**结论**：`load_pipeline` / notebook **无逻辑 bug**；问题来自 01 阶段 `parse_pmc_xml()` 的 XPath 设计。建议在 **§5 token 统计前** 修 parser 并重生成 jsonl；§3 可先基于现状做完整字段表并记录问题。

---

## 问题 1：`title` 异常过长、混入多篇文献标题

### 现象

- `title_char_len`：均值约 **4610** 字符，**96/100** 篇 > 500 字符，最长约 **21363** 字符。
- 正常 PMC 论文标题通常 < 300 字符。
- notebook 预览中 `retrieval_text` 开头为正确标题，后面紧接 `"Primary Ciliary Dyskinesia"`、`"Laterality defects..."` 等——实为**参考文献列表里的 `<article-title>`**。

### 原因

01 中 `parse_pmc_xml()` 使用：

```python
title = _text("//article-title")  # 匹配全文任意位置的 article-title
```

PMC 使用 **JATS XML**。除 `<front>` 里本篇标题外，`<back><ref-list>` 中每条参考文献也有 `<article-title>`（被引文献标题）。

XPath `//article-title` 会选中**所有**节点；`_text()` 用空格把它们**拼成一条字符串**，导致 `title` 字段变成「本篇标题 + 数十篇参考文献标题」。

01 notebook §5.4 打印 XML 前 80 行时已可见：第 5 行是正确标题，第 9、14、27 行等是 ref 中的 `<article-title>`——与污染现象一致。

### 影响

| 环节 | 影响 |
|---|---|
| **检索单元** `title + abstract` | `retrieval_text` 前半段是噪声，embedding 被无关标题稀释，检索准确率下降 |
| **Token 长度统计（§5）** | 若对 `retrieval_text` 计 token，P95/P99 严重偏大，**无法**据此定 chunk 策略 |
| **字段完整性（§3）** | `title` 非空率 100%，但**语义上不可用**，需在报告中区分「有值」与「可信」 |
| **说明文档** | 若不注明，易误判为「医学论文标题都特别长」 |

### 如何修复

**根因修复**：只取 `<front>` 内、本篇元数据下的标题，例如：

```python
# 推荐：限定 front，取第一个 title-group 下的 article-title
title = _text("//front//title-group/article-title[1]")
# 或更窄：
# title = _text("(//front//article-meta//title-group/article-title)[1]")
```

**验证**：对单篇 XML（如 `PMC8774754.xml`）解析后，`len(title)` 应在几十～三百字符量级；与改前数千字符对比。

**重跑**：用修正后的 `parse_pmc_xml()` 对 `01 验证模型/data/raw/extracted/`（或复制到 02 的 raw）批量重生成 `sample.jsonl`，再复制/覆盖到 `02 数据处理/data/processed/`。

### 推荐方案（优先）

1. 在 `02 数据处理/src/parse_pmc.py` 实现修正版 parser（从 01 迁入并改 XPath）。
2. 对现有 284 篇 XML 重跑 → 更新 `sample.jsonl`（100 篇子集或全量）。
3. 重新执行 notebook §2，确认 `title_char_len` 中位数回落到合理范围。
4. **§5 及以后**再基于 `title + abstract` 做 token 与分割设计。

**临时绕行**（若暂不重跑）：§5 仅对 **`abstract` 单独**做 token 分布；`title` 在文档中标注「待 parser 修复」。

---

## 问题 2：`pub_year` 出现 `"2022 2022"`、`"2023 2023 2023"` 等

### 现象

- **85/100** 条 `pub_year` 含重复或多余年份字符串。
- 示例：`2022 2022`、`2023 2023 2023`、`2023 2026 2023`。

### 原因

01 中使用：

```python
pub_year = _text("//pub-date/year")
```

JATS 中 `<pub-date>` 可出现多次：

- `<front>` 内：出版日期、电子版日期、收稿/修回日期等；
- `<ref-list>` 内：每条参考文献自己的出版年。

与 `title` 相同，全局 XPath 把所有 `<year>` **拼接**成一条，产生重复年份。

### 影响

| 环节 | 影响 |
|---|---|
| **「近 5 年」过滤** | 无法直接 `pub_year >= 2021`；需清洗或重抽 |
| **按年聚合统计** | 年份分布图失真 |
| **元数据展示** | 用户可见字段不规范 |

任务书字段为 `pub_date`，当前仅有 `pub_year` 且质量差——说明文档中需写清**字段映射与限制**。

### 如何修复

```python
# 推荐：只取 front 内文章级 pub-date 的第一个 year
pub_year = _text("//front//pub-date/year[1]")

# 更严谨：优先 epub / ppub
# pub_year = _text("//front//pub-date[@pub-type='epub']/year") or _text("//front//pub-date[@pub-type='ppub']/year")
```

若任务书要求完整 `pub_date`，可从同一 `<pub-date>` 节点组 `year/month/day` 拼 ISO 日期字符串，字段名 `pub_date`。

### 推荐方案

与问题 1 **同一轮 parser 修复**一并改 XPath，重跑 jsonl。§3 中对 `pub_year` 增加「清洗后唯一年份」检查（正则取第一个四位年或解析后整数）。

---

## 问题 3：`abstract` 缺失 3%（3/100 篇）

### 现象

| pmcid | journal |
|---|---|
| PMC12110669 | Current Issues in Molecular Biology |
| PMC12882845 | Metabolic Brain Disease |
| PMC12882890 | Nuclear Medicine and Molecular Imaging |

- 缺失率 **3%** > 任务书阈值 **1%**，触发清洗策略决策。
- `title`、`body` 仍存在；`has_abstract=False`。

### 原因

可能两类（需对单篇 XML 抽检确认）：

1. **源 XML 无结构化 abstract**（仅正文、或 abstract 非 `<abstract><p>` 结构），`//abstract//p` 抽不到文本。
2. **abstract 在其它标签内**（如 `trans-abstract`、`abstract[@abstract-type=...]`），当前 XPath 过窄。

这与 XPath 全局污染不同，属于**覆盖率**问题；若 XML 确实无摘要，则属正常缺失。

### 影响

| 环节 | 影响 |
|---|---|
| **RAG 检索块** | 任务书默认以摘要为检索单元；无 abstract 则无法形成有效 chunk |
| **缺失率指标** | 全量库若比例类似，清洗规则必须写进说明文档 |
| **样本量** | 丢弃后验证集为 97 篇，对 100 篇验证影响小 |

### 如何修复 / 处理

**策略 A（推荐，RAG 场景）**：pipeline 增加过滤 `has_abstract == True`，丢弃 3 篇；说明文档记录丢弃规则与数量。

**策略 B**：填充占位符 `"[NO ABSTRACT]"`——不推荐，会污染向量检索。

**策略 C**：放宽 XPath，例如 `//front//abstract//p` 或合并多种 abstract 类型后再判空；若仍空再丢弃。

### 推荐方案

1. §3 对 3 篇各打开 XML 确认是「真无摘要」还是「XPath 漏抽」。
2. 若为漏抽 → 改 parser；若为真无 → **采用策略 A 丢弃**。
3. 在 `load_pipeline` 或下游增加 `drop_missing_abstract=True` 可选参数，并写入 `sample_clean.jsonl`。

---

## 问题 4：`pmid` 未抽取（任务书字段缺失）

### 现象

- jsonl 中无 `pmid` 列；无法生成 `https://pubmed.ncbi.nlm.nih.gov/{pmid}/` 追溯链接。

### 原因

01 的 `parse_pmc_xml()` 未抽取：

```python
pmid = _text("//article-id[@pub-id-type='pmid']")
```

多数 PMC 全文 XML 在 `<front><article-meta>` 下含 `pub-id-type="pmid"` 的 `<article-id>`，属**功能遗漏**而非 XPath 污染。

### 影响

- 任务书「pmid 能否追溯原文」：**当前不能**。
- 可用 `pmcid` 链 PMC：`https://www.ncbi.nlm.nih.gov/pmc/articles/{PMCID}/` 作为替代。

### 推荐方案

与问题 1/2 **同轮 parser 修复**增加 `pmid` 字段；无 pmid 的篇保留空字符串并在 §3 统计覆盖率。

---

## 修复实施顺序（推荐）

```mermaid
flowchart LR
  A[修正 parse_pmc.py XPath] --> B[对 XML 批量重跑]
  B --> C[更新 sample.jsonl]
  C --> D[重跑 notebook §2 验证]
  D --> E[§3 字段完整性]
  E --> F[§5 token 用 title+abstract]
```

| 步骤 | 动作 | 产出 |
|---|---|---|
| 1 | `02/src/parse_pmc.py`：title / pub_year / pmid / abstract XPath | 可复用函数 |
| 2 | 输入：`01/data/raw/extracted/**/*.xml` | — |
| 3 | 输出：`02/data/processed/sample.jsonl`（及可选 `sample_clean.jsonl`） | 清洗后 97~100 篇 |
| 4 | `med-data-EDA.ipynb` §2 复跑 | title 长度正常、pub_year 单年 |
| 5 | 继续 §3~§6 | 说明文档与图表 |

**若时间紧**：先完成 §3（基于脏数据如实记录问题 + 清洗决策），再集中半天修 parser 并重跑；§5 前必须完成重跑或改用「仅 abstract」统计。

---

## 附录：诊断命令备忘

在 `02 数据处理` 根目录、环境 `med-rag-verify` 下：

```bash
cd "02 数据处理"
python -c "
import sys; sys.path.insert(0,'src')
from load_pipeline import resolve_project_dir, setup_paths, load_pmc_jsonl
pd,_=resolve_project_dir(); p=setup_paths(pd)
ds=load_pmc_jsonl(p['sample_jsonl'], backend='pandas', add_derived=True)
print('title>500:', (ds.title_char_len>500).sum(), '/', len(ds))
print('abstract缺失:', (~ds.has_abstract).sum())
"
```

相关文件：

- 计划：`02 数据处理/schedule.md` → 「数据质量问题」小节
- 分析：`02 数据处理/notebooks/med-data-EDA.ipynb`
- 01 原 parser：`01 验证模型/med-LLM-RAG.ipynb` §5.5

---

## 补充 Q&A

> 以下是对「数据问题最终怎么解决」「`src/` 各文件作用」「相对 01 继承/修改」「外接盘要设什么」的整理，便于与上文**问题 1~4 的历史诊断**对照回顾。

### Q1：针对之前的数据问题，最后采用什么方法解决？

分两层：

1. **根因（01 旧解析 XPath）** — title / pub_year 被全局 XPath 拼进参考文献；pmid 未抽。  
   **做法**：在 **`02 数据处理/src/parse_pmc.py`** 重写 XPath（限定 `//front//...`），用 **`build_jsonl.py`** + **`scripts/build_jsonl.sh`** 从 **XML 重新批量生成** `sample.jsonl`，不再依赖 01 notebook 里那段旧 `parse_pmc_xml()`。

2. **业务规则（非 parser bug）** — 约 **3%** 篇 XML 仍无可用 abstract。  
   **做法**：在 **`load_pipeline.py`** + notebook **§3** 中 **丢弃无 abstract**，得到 **`sample_clean.jsonl`（97 篇）** 供后续分析。

**一句话**：抽取规则在 **`parse_pmc.py`** 修；数据文件用 **`build_jsonl`** 重造；无摘要用 **清洗策略丢弃**。

---

### Q2：`02 数据处理/src/` 里各文件什么作用？

| 文件 | 作用 |
|---|---|
| **`parse_pmc.py`** | 单篇 PMC **JATS XML → dict**（title/abstract/body/journal/pub_year/pub_date/pmid/字符数）。**数据抽取质量的核心。** |
| **`build_jsonl.py`** | **CLI**：递归扫 XML 目录，调用 `parse_pmc_xml`，写 **JSONL**；支持 `--limit`、`--pmcids-from`、自动探测 XML 根目录。 |
| **`load_pipeline.py`** | **读 jsonl 做分析**：路径、`MED_RAG_DATA_ROOT`、`datasets`/`pandas`、派生列、§3 完整性/质量/元数据、`drop_missing_abstract` 等。**不负责从 XML 抽字段。** |
| **`domain_analysis.py`** | **阶段 4**（领域理解）骨架：abstract token 分层、IMRaD/缩写等；与「修脏数据」无关。 |

---

### Q3：解决「数据问题」主要在哪个代码文件？

- **改抽取逻辑**（title/pub_year/pmid 等）：**`parse_pmc.py`**。  
- **把修复落地成新 jsonl**：**`build_jsonl.py`**（`build_jsonl.sh` 只是调 Python）。  
- **`load_pipeline.py`** 只消费已生成好的 jsonl；若 jsonl 仍是 01 旧版，它无法消除 title 污染。

---

### Q4：相对 01 的数据处理，继承了什么、修改了什么？

**继承（思想）**

- 仍是 **lxml + XPath** 读 JATS，输出 **一行一篇 jsonl**。
- 字段与 01 目标一致，02 额外 **`pmid`、`pub_date`**。

**修改（相对 01 `med-LLM-RAG.ipynb` §5.5）**

| 点 | 01 | 02 |
|---|---|---|
| title | `//article-title` | `//front//title-group/article-title[1]` |
| pub_year | `//pub-date/year` 全局 | `front` 内 epub/ppub 等，取单年 |
| pmid | 无 | 有 |
| pub_date | 无 | 有 |
| 形态 | 写在 ipynb cell | **独立 `.py` + 可脚本批跑** |

---

### Q5：未来接外接盘全量数据，要改 / 设置什么？

**环境变量（优先，少改代码）**

- **`MED_RAG_DATA_ROOT`**：外接盘数据根，如 `/Volumes/盘名/med-rag-pmc`（其下 `extracted/`、`processed/` 与 schedule 约定一致）。
- **`PMC_XML_ROOT`**（可选）：若 XML 根不在默认推断路径，单独指定解压目录。

**命令示例**

```bash
export MED_RAG_DATA_ROOT=/Volumes/你的盘/med-rag-pmc
export PMC_XML_ROOT=/Volumes/你的盘/med-rag-pmc/extracted   # 需要时
cd "02 数据处理"
./scripts/build_jsonl.sh -o data/processed/oa_comm_full.jsonl
```

**代码里已预留**

- `build_jsonl.resolve_xml_root()`：按 `PMC_XML_ROOT` → `MED_RAG_DATA_ROOT/extracted` → 本工程 `data/raw/extracted` → 01 extracted 顺序探测。
- `load_pipeline.setup_paths()`：若设 `MED_RAG_DATA_ROOT`，读 jsonl 会切到该根下的 `processed/`。

**一般不必改 `parse_pmc.py`**，除非全量 XML 与当前子集结构差异大再微调 XPath。

---

*与 `schedule.md` 数据流、阶段 B 一致；以仓库当前代码为准。*

# PMC JATS 解析（parse_pmc.py）

## parse_pmc.py 如何从原 XML 抽取字段并输出？（举例说明）

**代码位置**：`02 数据处理/src/parse_pmc.py`，入口函数 `parse_pmc_xml(xml_path)`。

### 1. 整体在做什么

1. `lxml.etree.parse(xml_path)` 读入整篇 **JATS XML**，得到 `root`（根元素通常是 `<article>`）。
2. 对每条业务字段，用 **XPath 只在「该出现的区域」取节点**（多数限定在 `//front//...`，正文用 `//body//...`），避免把参考文献里的 `<article-title>`、`<year>` 拼进来（这是相对 01 旧版的核心修复）。
3. 用三个小工具把「节点 → 字符串」：
   - **`_first_text(root, xpath)`**：XPath 命中的**第一个**节点，把子树里所有文本拼成一段（用于 title、pmcid、单段 year 等）。
   - **`_join_text(root, xpath, sep)`**：XPath 可能命中**多个**节点（如多个 `<p>`），每个节点各自抽文本，再用 `sep`（默认换行）拼成一大段（用于 abstract、body）。
4. 组装成 **`dict`** 返回；`build_jsonl.py` 再按 `JSONL_FIELDS` 顺序 `json.dumps` 写成 **一行一条** 的 jsonl。

### 2. 举例：单篇 `PMC8774754.xml`（Diagnostics 那篇）

下面只列 **XML 里实际长什么样 → XPath → 输出字段** 的对应关系（原文可在 `01 验证模型/data/raw/extracted/PMC008xxxxxx/PMC8774754.xml` 打开对照）。

| 输出字段 | XPath / 逻辑 | XML 里典型位置（语义） | 本例会得到 |
|----------|----------------|------------------------|------------|
| `pmcid` | `_first_text(..., "//front//article-id[@pub-id-type='pmc']")` | `<front>…<article-meta>…<article-id pub-id-type="pmc">` | `PMC8774754` |
| `pmid` | `_first_text(..., "//front//article-id[@pub-id-type='pmid']")` | 同上，`pub-id-type="pmid"` | `35054295` |
| `title` | `_first_text(..., "//front//title-group/article-title[1]")` | 本篇标题，**不包含** `<back>` 里参考文献的 `<article-title>` | `Axonemal Symmetry Break, a New Ultrastructural Diagnostic Tool for Primary Ciliary Dyskinesia?` |
| `journal` | `_first_text(..., "//front//journal-title")` | `<journal-meta>` 下期刊名 | `Diagnostics` |
| `pub_year` | `_extract_pub_year`：依次试 epub / ppub / collection 的 `year`，再正则取四位年 | `<pub-date pub-type="epub"><year>2022</year>` 等 | `2022` |
| `pub_date` | `_extract_pub_date`：取首个 epub 或 ppub 的 `year/month/day` 拼成 ISO | epub 为 2022-01-06 | `2022-01-06` |
| `abstract` | 先 `_join_text(..., "//front//abstract[not(@abstract-type='graphical')]//p")`；若仍空再放宽到整块 `abstract` | `<front>…<abstract><p>…</p></abstract>` | 一段完整英文摘要（多 `<p>` 会换行拼接） |
| `body` | `_join_text(..., "//body//p")` | `<body>` 下各节里所有 `<p>`，**不按 sec 分块**，只按段落顺序用换行拼 | 很长的一串正文文本 |
| `n_chars_*` | `len(abstract)`、`len(body)` | — | 字符数 |

**输出形态**：Python 里是一个 **普通 dict**，例如：

```python
{
  "pmcid": "PMC8774754",
  "pmid": "35054295",
  "title": "Axonemal Symmetry Break, ...?",
  "abstract": "Diagnosis testing for primary ciliary ...",
  "body": "Primary ciliary dyskinesia (PCD) is ...\n...",
  "journal": "Diagnostics",
  "pub_year": "2022",
  "pub_date": "2022-01-06",
  "n_chars_abstract": 1243,  # 示例数，以实际 len 为准
  "n_chars_body": 13813,
}
```

`build_jsonl.py` 会把它变成 **jsonl 一行**（UTF-8、`ensure_ascii=False`），字段顺序由模块顶部 **`JSONL_FIELDS`** 元组固定。

### 3. 和「只输出」相关的注意点

- **不做清洗决策**：例如「无 abstract 的篇是否丢弃」在 **`load_pipeline` + notebook §3** 里做，`parse_pmc` 只负责如实抽字符串。
- **正文很大**：`body` 会把所有 `//body//p` 拼在一起，行数多、体积大是预期行为；检索单元设计在分析阶段用 `title+abstract` 为主。

### 4. 想自己复现一条

在 `02 数据处理` 下、已激活 `med-rag-verify`：

```python
from pathlib import Path
import sys
sys.path.insert(0, "src")
from parse_pmc import parse_pmc_xml

p = Path("../01 验证模型/data/raw/extracted/PMC008xxxxxx/PMC8774754.xml")
rec = parse_pmc_xml(p)
print(rec["pmcid"], rec["pmid"], len(rec["title"]), len(rec["abstract"]))
```

# 02 阶段在整体链路中的位置（相对 RAG）

## 1. 现在做的「数据处理」是不是为 RAG 先整理数据、方便 LLM 检索？

**大体对，建议再精确一点：**

| 说法 | 是否准确 |
|------|----------|
| 在搭 RAG **之前** 先处理 PMC 数据 | ✅ |
| 目的是让后面 LLM **检索时** 用的文本更可靠、有依据 | ✅（间接目的） |
| 当前阶段已经在做 **向量检索 / 问答** | ❌（02 明确不做 Chroma 入库、不做 LangChain 链） |

更准确的一句话：

> **02 做的是「数据 pipeline + 质量/领域/长度评估 + 分割方案设计」**，交付《RAG 数据分析与设计说明》；  
> 相当于 RAG 开工前的 **摸底 + 定规矩**，不是 RAG 本体。

可以画成两条线：

```text
【你现在在做的 02】
  XML → parse_pmc → jsonl（结构化）
       → 字段/质量分析（§1~3）
       → 领域理解 + token 分布 + 分割策略（§4~6，进行中）
       → 说明文档（给后面工程看）

【之后的 RAG 阶段（LangChain_RAG 等）】
  按说明文档选字段、chunk、embedding
       → 写入向量库
       → 用户提问 → 检索相关 chunk → LLM 生成答案
```

所以：**是在为 RAG「铺路」**，但铺路的内容主要是 **「抽什么、丢什么、怎么切、切多大」**，而不是已经帮 LLM「整理好可直接检索的库」。

---

## 2. body 很大，全量 jsonl 会不会和原始数据差不多大？后面 §2~§4 是为了「简化内容、方便定位、缩小体积」吗？

### 2.1 体积：你的直觉对

- 当前 `parse_pmc` 把 **`body` 全文** 写进 jsonl（所有 `//body//p` 换行拼接）。
- 验证期 100 篇里，**单篇 body 字符数往往是 abstract 的十几～几十倍**；全量若每篇都保留 body，**jsonl 总体积不会比「只存 XML」小很多**（JSON 转义后甚至可能略大）。
- 全量 PMC 是 **百 GB 级原始包**；若全库 jsonl 且每行带完整 body，**外接盘仍然要很大**，不能只靠「转成 jsonl」省空间。

### 2.2 任务书 §2 / §3 / §4 各自在干什么？（≠ 同一类「压缩」）

| 阶段 | 任务书名称 | **主要目的** | **会不会明显缩小全量 jsonl？** |
|------|------------|--------------|--------------------------------|
| **§2** | 领域内容理解 | 懂医学摘要的**风格、术语、结构**；分层抽 5~10 篇做**人工阅读**；给 prompt/评估定「心理基线」 | **基本不会**。产出是 `stratified_*.md`、统计表，不是替换全库正文 |
| **§3** | 文本特征量化（token） | 用 **embedding 同款 tokenizer** 看 title/abstract/body 的 **P50/P95/P99**；判断要不要切、切多大 | **不会**。是**统计**，不是删字段 |
| **§4** | 制定文本分割策略 | 写进说明文档：**检索单元用 title+abstract 还是也含 body**；长尾用滑动窗口还是按章节 | **设计阶段不缩库**；真正切分在 **RAG 建库时** 才发生 |

**结论（针对你的猜想）：**

- §2~§4 **不是**「把全量 jsonl 简化成更小文件给 LLM 快速定位」的流水线步骤。  
- 它们主要是 **分析与设计**，让后面 RAG **知道怎么建索引**，而不是现在就把全库变瘦。

### 2.3 那 LLM「快速定位」靠的是什么？

RAG 里「快速定位」靠的是 **向量检索（或关键词检索）**，不是老师这份任务书里的 §2~§4 本身：

1. **建库时**：把文档切成 **chunk**（每块几百 token），每块算一个 **embedding** 放进向量库。  
2. **提问时**：只算 **问题的 embedding**，在库里找 **Top-K 个最相似的 chunk**。  
3. **生成时**：LLM 只读这 K 块文本（+ 可选元数据），**不会每次加载整篇 body**。

所以：**缩小「每次进 LLM 的文本」** 靠的是 **chunk + 检索**，这是 **下一阶段 RAG** 的事；02 的 §4 只是 **事先定 chunk 规则**。

### 2.4 若要在全量期「省存储」，可以怎么做？（与 02 分析的关系）

| 策略 | 说明 | 和当前 02 的关系 |
|------|------|------------------|
| **检索单元不含 body** | 常见做法：入库只用 `title + abstract`（你们口径已倾向这个）；body 仅追溯或二期再做 | §3/§4 用数据证明「摘要大多 <512 token」即可支持 |
| **jsonl 分层** | 例如 `metadata.jsonl`（无 body）+ `body` 仍放 XML 或按需解析 | parse 逻辑可拆两种输出，属工程优化，非任务书必须 |
| **建库时再切 body** | 全量 jsonl 可仍很大，但 **向量库** 存的是 chunk，单条比整篇 body 小；总 chunk 数 × 维度才是向量库体积 | §4 定 `chunk_size` / `overlap` |
| **不全量 jsonl 化** | 流式：下载 → 解析 → **直接 chunk + embedding** → 丢中间全文 | 全量期 schedule 阶段 B 可演进 |

**重要区分：**

- **jsonl（结构化存档）**：方便 pandas/datasets 做统计；**不必等于 RAG 最终入库形态**。  
- **向量库（检索用）**：才是 LLM 检索时「快速定位」的对象；体积由 **chunk 数量 × embedding 维度** 决定。

因此：**§4 分割策略** 的意义是 —— 将来若要对 body 建索引，**按什么规则切成小块**；它**不能**在 02 阶段就把「全量抽取 body 的 jsonl」变得和「只存 abstract」一样小，除非你 **决定 jsonl 里根本不存 body**。

---

## 3. 后续各步（结合本仓库）一句话意义

| 步骤 | 意义（给 RAG 什么） |
|------|---------------------|
| 已完成：parse + jsonl | 统一、可统计的表结构；修复 title 等字段 |
| 已完成：§1 结构与清洗 | 哪些篇能进库（如丢弃无 abstract） |
| 待做：§2 领域理解 | **人**理解医学文本长什么样，写进说明文档 |
| 待做：§3 token 分布 | 用数字回答「要不要切、切多大」 |
| 待做：§4 分割策略 | 写清 **chunk 方案**（摘要一整块 vs 正文滑动窗口） |
| 未来：RAG 工程 | 按说明文档 **embedding + 向量库 + LLM** |

**记忆口诀**：02 = **体检报告 + 施工图纸**；RAG = **按图纸盖房子（向量库 + 问答）**。

# 全量 jsonl 体积、清洗与 slim 方案（合并 Q&A）

> 合并自 2026-05 两轮讨论；工程约定已写入 `02 数据处理/schedule.md` 阶段 B，代码见 `build_jsonl.py --slim`、`pmc_index.py`。

## 1. 按当前做法，全量会生成很大的 jsonl 吗？会存在两份巨大 jsonl 吗？

**会很大（若每行带完整 body）。** 验证期已见：单篇 `body` 常为 abstract 的十几～几十倍；全库 jsonl 若含 body，体积与「把所有 XML 正文再抄进 JSON」同级，**不能指望比原始包小**。

**两份巨大 jsonl 不是必须：**

| 文件 | 验证期 | 全量期（已定方案） |
|------|--------|-------------------|
| `sample.jsonl` | 100 篇，**含 body** | 不使用 |
| `sample_clean.jsonl` | 97 篇，几乎整份复制少 3 行 | **不生成** |
| `oa_comm_slim.jsonl` | — | **唯一主表**：无 body 正文，解析阶段已丢弃无 abstract |

验证期保留 `sample` + `sample_clean` 是为 §3 交付可核对；全量期 **一份 slim + `skipped_no_abstract.txt`（审计用，很小）** 即可。

---

## 2. 分割策略：只给 embedding/向量库打底，还是能用于全量？

| 层次 | 02 阶段 | RAG 阶段 |
|------|---------|----------|
| §4 产出 | **设计说明**（chunk 规则、参数） | — |
| 执行 | notebook 可 **demo 2～3 篇** | **全量**按说明对允许入库的篇目切 chunk → embedding → 向量库 |

分割策略 **要在全量建库时执行**；02 负责用 100 篇（+ 可选抽样）把参数定准。它 **缩小的是每次检索进 LLM 的 chunk**，不是 slim jsonl 文件本身。

---

## 3. §1 不要求 body，全量 jsonl 能否不存 body？用 pmcid 回查 XML？

**可以，且推荐。** 任务书 §1 是字段完整性、质量、元数据——**未要求** jsonl 存 body 全文。

| 阶段 | 是否需要 jsonl 里的 body |
|------|--------------------------|
| §1 结构分析 | 否；`n_chars_body` / `has_body` 在 parse 时即可写入 slim 行 |
| §2 领域理解 | 否；分层抽样读 **abstract** |
| §3 token（摘要） | 否；title / abstract / title+abstract |
| §3 token（正文） | 否；抽样或流式扫 **XML**，不必全库 body 进 jsonl |
| §4 分割策略 | 否；摘要策略读 slim；正文策略用抽样 + 01 均长经验 |

**回查方式：**

- 主键用 **`pmcid`**（不是 pmid：约 7% 缺失，且 pmid 不能直接推出 XML 路径）。
- 路径规则：`PMC{数字//1_000_000:03d}xxxxxx/PMC{id}.xml`（见 `parse_pmc.resolve_pmc_xml_path`）。
- 全量前 **`build_pmcid_index.sh`** 扫一遍生成 `pmcid_index.jsonl`，避免百万次 `glob`。
- 按需读正文：`pmc_index.load_body_for_pmcid(pmcid)`。

**能省多少空间？** slim jsonl 主要为元数据 + 摘要，通常 **远小于** 带 body 的 jsonl；XML 仍占一份（权威全文），但避免 **第二份正文副本**。

---

## 4. 全量清洗：parse 阶段直接丢弃无 abstract

验证期结论：abstract 缺失 3% > 任务书 1% 阈值 → **丢弃**。

全量期不在事后写 `*_clean.jsonl`，而在构建时：

```bash
./scripts/build_full_slim.sh
# 等价于 build_jsonl.sh --slim --skip-no-abstract
```

无 abstract 的篇目不写入 `oa_comm_slim.jsonl`，`pmcid` 记入 `skipped_no_abstract.txt` 备查。

---

## 5. 环境变量速查（全量期）

```bash
export MED_RAG_DATA_ROOT=/Volumes/<盘>/med-rag-pmc
export PMC_XML_ROOT=$MED_RAG_DATA_ROOT/extracted
export MED_RAG_JSONL=$MED_RAG_DATA_ROOT/processed/oa_comm_slim.jsonl

./scripts/build_pmcid_index.sh
./scripts/build_full_slim.sh
# notebook 前置 cell 会通过 load_pipeline 读取 MED_RAG_JSONL
```

**记忆口诀**：验证期 = 带 body 的小样本 + 可留 clean 副本；全量期 = **XML 一份全文 + slim 一张表 + 索引定位 + parse 时丢掉无摘要**。

## 当前修改：
已完成三项工作，摘要如下。

---

### 1. 笔记 `笔记/02笔记.ipynb`

文末新增 **`# 全量 jsonl 体积、清洗与 slim 方案（合并 Q&A）`**，合并两轮讨论：

- 全量 jsonl 体积、不必双份 clean
- 分割策略在 02 vs RAG 的分工
- slim jsonl + `pmcid` 回查 XML
- parse 阶段丢弃无 abstract
- 全量环境变量速查

---

### 2. 计划 `02 数据处理/schedule.md`

重写 **阶段 B / 全量期工程约定**，包括：

- 验证期 `sample.jsonl`（含 body）vs 全量 `oa_comm_slim.jsonl`
- 外接盘目录、`pmcid_index`、`skipped_no_abstract.txt`
- 不再生成全量 `*_clean.jsonl`
- 阶段 B 六步工作流
- 目录树与阶段 2  checklist 已勾选新脚本能力

---

### 3. 工程改动（为全量做准备）

| 文件 | 改动 |
|------|------|
| `src/parse_pmc.py` | `JSONL_FIELDS_SLIM`、`resolve_pmc_xml_path`、`record_for_jsonl`、`has_abstract` |
| `src/pmc_index.py` | **新建**：建索引、按 pmcid 读 body |
| `src/build_jsonl.py` | `--slim`、`--skip-no-abstract`、`--skipped-log` |
| `src/load_pipeline.py` | full/slim 列集、`infer_jsonl_mode`、`MED_RAG_JSONL`、无 body 时用 `n_chars_body` |
| `scripts/build_jsonl.sh` | 注释更新 |
| `scripts/build_full_slim.sh` | **新建**：全量 slim 一键 |
| `scripts/build_pmcid_index.sh` | **新建**：预建 pmcid 索引 |
| `docs/RAG数据分析与设计说明.md` | 验证期 vs 全量 slim 说明 |
| `README.md` | 全量脚本与数据表说明 |

**验证期不变**：仍用带 body 的 `sample.jsonl` + `sample_clean.jsonl`。

**全量期命令示例**：

```bash
export MED_RAG_DATA_ROOT=/Volumes/<盘>/med-rag-pmc
export PMC_XML_ROOT=$MED_RAG_DATA_ROOT/extracted
export MED_RAG_JSONL=$MED_RAG_DATA_ROOT/processed/oa_comm_slim.jsonl

./scripts/build_pmcid_index.sh
./scripts/build_full_slim.sh
```

本地已对 `--slim` 做过 5 篇 smoke test：输出无 `body` 键，保留 `n_chars_body`。

---

100 篇验证跑完后，只需改环境变量并执行上述脚本，notebook 会通过 `MED_RAG_JSONL` 加载 slim 表，无需改 §3 分析逻辑。